<a href="https://colab.research.google.com/github/fasih779/Fine-Tuning-LL/blob/main/PPC_and_CrpC_LLM_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**YOU CAN ASK ANY LAW QUESTION RELATED TO CRPC OR PPC IT SIMPLIFY IT TO YOUR UNDERSTANDING**

In [40]:
!pip install -U transformers trl peft bitsandbytes accelerate datasets torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

In [4]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "qwen2.5-3b-legal-simplification"
from datasets import load_from_disk

dataset = load_from_disk("pak_legal_sft")

In [6]:
def convert_to_chat_format(example):
    if example["input"] and example["input"].strip():
        user_content = example["instruction"] + "\n" + example["input"]
    else:
        user_content = example["instruction"]

    return {
        "messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": example["output"]}
        ]
    }

dataset = dataset.map(convert_to_chat_format, remove_columns=["instruction", "input", "output"])
dataset["train"][0]

Map:   0%|          | 0/686 [00:00<?, ? examples/s]

Map:   0%|          | 0/172 [00:00<?, ? examples/s]

{'messages': [{'role': 'user',
   'content': 'Simplify the following legal text into plain English.\nSection 147 of the Pakistan Penal Code (PPC): Punishment for rioting:\n\nWhoever is guilty of rioting, shall be punished with imprisonment of either description for a term which may extend to two years, or with fine, or with both.'},
  {'role': 'assistant',
   'content': 'Section 147 of the Pakistan Penal Code (PPC): Punishment for rioting:\n\nWhoever is guilty of rioting, shall be punished with imprisonment of either description for a term which may extend to two years, or with fine, or with both.'}]}

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
print("EOS token:", tokenizer.eos_token)
print("Chat template present:", tokenizer.chat_template is not None)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


EOS token: <|im_end|>
Chat template present: True


In [17]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [19]:
# Get lengths paired with their index so we can inspect the long ones
lengths_with_idx = [
    (i, len(tokenizer(tokenizer.apply_chat_template(ex["messages"], tokenize=False))["input_ids"]))
    for i, ex in enumerate(dataset["train"])
]

# Sort descending by length
lengths_with_idx.sort(key=lambda x: x[1], reverse=True)

# Show the top 10 longest examples
for idx, length in lengths_with_idx[:10]:
    print(f"Index {idx}: {length} tokens")
worst_idx = lengths_with_idx[0][0]
print(dataset["train"][worst_idx]["messages"][0]["content"][:1000])  # first 1000 chars of user message
print("---")
print(dataset["train"][worst_idx]["messages"][1]["content"][:1000])  # first 1000 chars of assistant message
import numpy as np

lengths_only = [l for _, l in lengths_with_idx]
print("50th percentile:", np.percentile(lengths_only, 50))
print("90th percentile:", np.percentile(lengths_only, 90))
print("95th percentile:", np.percentile(lengths_only, 95))
print("99th percentile:", np.percentile(lengths_only, 99))

Index 682: 34541 tokens
Index 511: 9534 tokens
Index 626: 4206 tokens
Index 353: 3803 tokens
Index 591: 3700 tokens
Index 651: 3112 tokens
Index 617: 2733 tokens
Index 581: 1987 tokens
Index 666: 1987 tokens
Index 78: 1961 tokens
Simplify the following legal text into plain English.
Section 295C of the Code of Criminal Procedure (CrPC): Use of derogatory Ditto .. Ditto .. Ditto .. Ditto .. Death, or Court of

remarks, etc., in imprisonment Session respect of the for life and which shall Holy Prophet. fine. be presided over by a Muslim.] 296 Causing a Imprisonment [May [Summons] [Bailable] [Not [ [* * *] disturbance to an of either arrest compound Magistrate assembly engaged description without -able.] of the first in religious for one warrant.] or second worship. year or fine, class.] or both. Ditto .. Ditto .. Ditto .. Ditto .. Ditto .. 297 Trespassing in Imprisonment place of worship or of either sepulture,disturbing description funeral with inten for one year, tion to wound the or f

In [10]:
lengths = [
    len(tokenizer(tokenizer.apply_chat_template(ex["messages"], tokenize=False))["input_ids"])
    for ex in dataset["train"]
]
print("Max length:", max(lengths))
print("Avg length:", sum(lengths) / len(lengths))

Max length: 34541
Avg length: 457.05102040816325


In [28]:
peft_config = LoraConfig(
    r=6,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

In [29]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/tmp/ipykernel_22787/3149180711.py:1: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(


In [30]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
)


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [33]:

max_seq_length = 512
def filter_long_samples(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return len(tokenizer.encode(text)) <= max_seq_length

dataset["train"] = dataset["train"].filter(filter_long_samples)
dataset["test"] = dataset["test"].filter(filter_long_samples)


sft_config.max_seq_length = max_seq_length
sft_config.per_device_train_batch_size = 2
sft_config.gradient_checkpointing = True
sft_config.fp16 = True

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
)

print(f"Training with {len(dataset['train'])} samples after filtering (max_len=512).")
trainer.train()

Filter:   0%|          | 0/651 [00:00<?, ? examples/s]

Filter:   0%|          | 0/166 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/551 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/140 [00:00<?, ? examples/s]

Training with 551 samples after filtering (max_len=512).


Step,Training Loss
10,1.370073


TrainOutput(global_step=18, training_loss=1.1323738892873128, metrics={'train_runtime': 1975.9867, 'train_samples_per_second': 0.279, 'train_steps_per_second': 0.009, 'total_flos': 2888740913516544.0, 'train_loss': 1.1323738892873128, 'entropy': 0.8295164324086288, 'mean_token_accuracy': 0.8217754225278723, 'num_tokens': 142532.0, 'epoch': 1.0})

In [34]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to:", OUTPUT_DIR)

Saved to: qwen2.5-3b-legal-simplification


In [35]:
!pip install -U gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: starlette
    Found existing installation: starlette 0.52.1
    Uninstalling starlette-0.52.1:
      Successfully uninstalled starlette-0.52.1
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.3.1 whi

In [2]:
import torch
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_PATH = "qwen2.5-3b-legal-simplification"

In [4]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
basemodel = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="cuda",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
# Loading the fine-tuned adapter
model = PeftModel.from_pretrained(basemodel, ADAPTER_PATH)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [5]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=6, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=6, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
 

In [9]:
def simplify_legal_text(instruction, legal_text, max_new_tokens=256, temperature=0.7):
    if legal_text.strip():
        user_content = instruction + "\n" + legal_text
    else:
        user_content = instruction

    messages = [{"role": "user", "content": user_content}]


    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    return response

In [ ]:
demo = gr.Interface(
    fn=simplify_legal_text,
    inputs=[
        gr.Textbox(
            label="Instruction",
            value="Simplify the following legal text into plain English.",
            lines=2
        ),
        gr.Textbox(
            label="Legal Text (input)",
            placeholder="Paste a section of the Pakistan Penal Code here...",
            lines=8
        ),
        gr.Slider(minimum=32, maximum=512, value=256, step=16, label="Max New Tokens"),
        gr.Slider(minimum=0.1, maximum=1.5, value=0.7, step=0.1, label="Temperature"),
    ],
    outputs=gr.Textbox(label="Simplified Output", lines=8),
    title="Legal Text Simplifier (Qwen2.5-3B + QLoRA)",
    description="Fine-tuned on Pakistan Penal Code simplification dataset.",
)

demo.launch(share=True, debug=True) # share=True gives you a public link from Colab

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://804d68440a84fb6dbf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Using existing dataset file at: .gradio/flagged/dataset1.csv


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


In [1]:
!pip uninstall -y torchao
!pip install -U torchao

Found existing installation: torchao 0.17.0
Uninstalling torchao-0.17.0:
  Successfully uninstalled torchao-0.17.0
  Using cached torchao-0.17.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (20 kB)
Using cached torchao-0.17.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (3.2 MB)
